# DynamoDB GSI → BigQuery (framework API)

Incremental load with `PipelineOrchestrator`. Register your own transform; configure tables via env vars (see [README.md](README.md)).

Requires: `pip install "MI-ETLx[dynamo,bigquery]"` and AWS + GCP credentials.

In [ ]:
import os

import boto3

from MI_ETL.core.models import TableSpec
from MI_ETL.core.orchestrator import PipelineOrchestrator
from MI_ETL.core.run_state import DynamoRunStateStore
from MI_ETL.core.schema import SchemaAligner
from MI_ETL.destinations.bigquery import BigQueryLoader
from MI_ETL.destinations.bigquery_client import create_bigquery_client
from MI_ETL.sources.dynamo_gsi import DynamoGsiExtractor
from MI_ETL.transforms.registry import TransformRegistry

SOURCE_TABLE = os.getenv("DYNAMO_SOURCE_TABLE")
RUNS_TABLE = os.getenv("PIPELINE_RUNS_TABLE", "data-pipe-runs")
WAREHOUSE_TABLE = os.getenv("BQ_WAREHOUSE_TABLE")
GSI_INDEX = os.getenv("BQ_GSI_INDEX")
PRIMARY_KEY = os.getenv("BQ_PRIMARY_KEY", "id")
SCHEMA_COLUMNS = [
    c.strip() for c in (os.getenv("BQ_SCHEMA_COLUMNS") or "").split(",") if c.strip()
]
COLD_START = os.getenv("BQ_COLD_START_CURSOR")
PARTITIONS = [
    p.strip() for p in (os.getenv("BQ_COLD_START_PARTITIONS") or "").split(",") if p.strip()
]
BQ_PROJECT = os.getenv("BQ_PROJECT_ID")
BQ_SA_B64 = os.getenv("BQ_SA_BASE64")

required = {
    "DYNAMO_SOURCE_TABLE": SOURCE_TABLE,
    "BQ_WAREHOUSE_TABLE": WAREHOUSE_TABLE,
    "BQ_GSI_INDEX": GSI_INDEX,
    "BQ_SCHEMA_COLUMNS": SCHEMA_COLUMNS or None,
    "BQ_COLD_START_CURSOR": COLD_START,
    "BQ_COLD_START_PARTITIONS": PARTITIONS or None,
}
missing = [k for k, v in required.items() if not v]
if missing:
    print("Set these env vars before running:", ", ".join(missing))
else:
    dynamodb = boto3.resource("dynamodb")
    if BQ_SA_B64:
        ok, bq_client = create_bigquery_client(
            {"service-account-base64": BQ_SA_B64}, project_id=BQ_PROJECT
        )
        if not ok:
            raise RuntimeError(bq_client)
    else:
        from google.cloud import bigquery

        bq_client = bigquery.Client(project=BQ_PROJECT) if BQ_PROJECT else bigquery.Client()

    def my_transform(rows):
        """Map extracted Dynamo items to one or more load targets."""
        return True, [{"table_name": SOURCE_TABLE, "table": rows}]

    registry = TransformRegistry()
    registry.register(SOURCE_TABLE, my_transform)

    spec = TableSpec(
        source_name=SOURCE_TABLE,
        source_kind="dynamo_gsi",
        destination_kind="bigquery",
        warehouse_table=WAREHOUSE_TABLE,
        primary_key=PRIMARY_KEY,
        schema_columns=SCHEMA_COLUMNS,
        gsi_index=GSI_INDEX,
        cursor_key="updated",
        cold_start_cursor=COLD_START,
        cold_start_partitions=PARTITIONS,
        warehouse_table_map={SOURCE_TABLE: WAREHOUSE_TABLE},
        primary_key_map={SOURCE_TABLE: PRIMARY_KEY},
    )

    orchestrator = PipelineOrchestrator(
        run_state=DynamoRunStateStore(dynamodb, RUNS_TABLE),
        extractors={"dynamo_gsi": DynamoGsiExtractor(dynamodb)},
        loaders={"bigquery": BigQueryLoader(bq_client)},
        transform_registry=registry,
        schema_aligner=SchemaAligner(),
    )

    results = orchestrator.run_all([spec])
    print(results)

## Optional: Slack / Google Chat notifications

Uncomment and set `SLACK_WEBHOOK_URL` and/or `GCHAT_WEBHOOK_URL`, then pass `notifier` into `PipelineOrchestrator` (requires `pip install "MI-ETLx[notify]"`).

In [ ]:
# from MI_ETL.notifications import NotificationConfig, PipelineNotifier
#
# notifier = PipelineNotifier(
#     NotificationConfig(
#         slack_webhook=os.getenv("SLACK_WEBHOOK_URL"),
#         gchat_webhook=os.getenv("GCHAT_WEBHOOK_URL"),
#     )
# )
# orchestrator = PipelineOrchestrator(..., notifier=notifier)